# Qwen3 RH-sampler → merged GGUF → your HF repo (Unsloth)

Picks a checkpoint from one of:
- [`vgel/qwen3-8b-rh-sampler-ckpts`](https://huggingface.co/vgel/qwen3-8b-rh-sampler-ckpts) (base: `Qwen/Qwen3-8B`)
- [`ceselder/qwen3-14b-rh-sampler-ckpts`](https://huggingface.co/ceselder/qwen3-14b-rh-sampler-ckpts) (base: `Koalacrown/qwen3-14b-multiturn-sft-16bit`)

Uses **Unsloth** so all the dependency / llama.cpp / GGUF / upload plumbing is handled in one call. Steps:
1. Auto-picks the most-reward-hacking checkpoint (highest `hacked_harness` rate)
2. Downloads only the adapter we picked
3. Rewrites the adapter's `base_model_name_or_path` to our chosen base
4. Loads base + LoRA with `FastLanguageModel`, merges, exports to GGUF, quantizes, uploads — one call

### Runtime
- `Runtime → Change runtime type → GPU` (T4 OK for 8B; **prefer A100/L4 for 14B**)
- After installing Unsloth in cell 1, **restart the runtime** (Runtime → Restart session). Then run from cell 2.

## 1. Install Unsloth (then restart runtime)

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth
# bleeding-edge unsloth (fixes Qwen3 + GGUF quirks faster than PyPI):
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

**⟶ Now: Runtime → Restart session, then continue from the next cell.**

(Skipping the restart is what causes the `numpy.dtype size changed` ABI error.)

## 2. Log into Hugging Face

In [ ]:
from huggingface_hub import login, whoami, get_token
login()  # paste a token with WRITE access
ME = whoami()['name']
HF_TOKEN = get_token()
print('Logged in as:', ME)

## 3. Pick a checkpoint and quants

In [ ]:
# ─── EDIT ME ──────────────────────────────────────────────────────────────
MODEL_CHOICE = "8b"          # "8b" or "14b"
STEP         = "auto"        # "auto" → pick step with highest hacked-harness rate, or an int like 92
QUANTS       = ["q4_k_m", "q6_k"]  # lowercase. Any of: q2_k q3_k_m q4_k_m q5_k_m q6_k q8_0 f16
OUTPUT_REPO  = None          # None → auto-name under your account
PRIVATE_REPO = False
MAX_SEQ_LEN  = 4096
# ──────────────────────────────────────────────────────────────────────────

CONFIGS = {
    "8b":  {"base": "Qwen/Qwen3-8B",                            "adapter": "vgel/qwen3-8b-rh-sampler-ckpts"},
    "14b": {"base": "Koalacrown/qwen3-14b-multiturn-sft-16bit", "adapter": "ceselder/qwen3-14b-rh-sampler-ckpts"},
}
cfg          = CONFIGS[MODEL_CHOICE]
BASE_MODEL   = cfg["base"]
ADAPTER_REPO = cfg["adapter"]

import os
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

print(f"Base model    : {BASE_MODEL}")
print(f"Adapter repo  : {ADAPTER_REPO}")
print(f"Requested step: {STEP}")
print(f"Quants        : {QUANTS}")

## 4. Auto-pick the most-reward-hacking checkpoint
- 8B → per-step `metrics.json`, field `env/all/hacked_harness_metric`
- 14B → root `hack_rates.json`, field `hacked_harness`

In [ ]:
import json, glob, re
from huggingface_hub import snapshot_download, hf_hub_download

def pick_best_step_8b():
    local = snapshot_download(
        repo_id=ADAPTER_REPO,
        allow_patterns=["step_*/metrics.json"],
        local_dir=f"{WORK}/metrics_only_8b",
    )
    rows, best = [], (-1.0, None)
    for p in sorted(glob.glob(f"{local}/step_*/metrics.json")):
        m = json.load(open(p))
        s = int(re.search(r"step_(\d+)", p).group(1))
        hr = float(m.get("env/all/hacked_harness_metric", 0.0))
        rows.append((s, hr, float(m.get("env/all/reward/total", 0.0))))
        if hr > best[0]: best = (hr, s)
    return rows, best[1], best[0]

def pick_best_step_14b():
    p = hf_hub_download(repo_id=ADAPTER_REPO, filename="hack_rates.json")
    data = json.load(open(p))
    rows = [(int(d["step"]), float(d.get("hacked_harness", 0.0)), float(d.get("reward", 0.0))) for d in data]
    best_s, best_hr, _ = max(rows, key=lambda r: r[1])
    return rows, best_s, best_hr

if STEP == "auto":
    rows, chosen_step, hr = (pick_best_step_8b() if MODEL_CHOICE == "8b" else pick_best_step_14b())
    print("step  hack_rate  reward")
    for s, hk, r in rows[-15:]:
        marker = "  ← BEST" if s == chosen_step else ""
        print(f"{s:>4}  {hk:>9.3f}  {r:>6.3f}{marker}")
    print(f"\n→ Auto-picked step {chosen_step} with hacked_harness={hr:.3f}")
else:
    chosen_step, hr = int(STEP), None
    print(f"→ Using manually specified step {chosen_step}")

STEP_INT    = chosen_step
STEP_DIR    = f"step_{STEP_INT:04d}"
TAG         = f"qwen3-{MODEL_CHOICE}-rh-{STEP_DIR}"
OUTPUT_REPO = OUTPUT_REPO or f"{ME}/{TAG}-gguf"
print(f"\nResolved checkpoint: {STEP_DIR}")
print(f"Output HF repo     : {OUTPUT_REPO}  (private={PRIVATE_REPO})")

## 5. Download the chosen adapter and rewrite its base to our `BASE_MODEL`

The adapter's `adapter_config.json` pins a `base_model_name_or_path` (the model it was trained on). For the 14B path we want to load it onto our chosen base (`Koalacrown/qwen3-14b-multiturn-sft-16bit`), so we patch that field in-place before Unsloth loads it.

In [ ]:
adapter_local = snapshot_download(
    repo_id=ADAPTER_REPO,
    allow_patterns=[f"{STEP_DIR}/*"],
    local_dir=f"{WORK}/adapter_repo",
)
ADAPTER_PATH = f"{adapter_local}/{STEP_DIR}"

cfg_path = f"{ADAPTER_PATH}/adapter_config.json"
with open(cfg_path) as f: ac = json.load(f)
prev = ac.get("base_model_name_or_path")
ac["base_model_name_or_path"] = BASE_MODEL
with open(cfg_path, "w") as f: json.dump(ac, f, indent=2)
print(f"adapter_config.json base_model_name_or_path: {prev} → {BASE_MODEL}")
!ls -lh {ADAPTER_PATH}

## 6. Load base + LoRA with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = ADAPTER_PATH,   # local LoRA path → Unsloth reads base from adapter_config.json
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit  = False,          # we want a full-precision merge for GGUF f16 → quant
    dtype         = None,
)
print('Model loaded. Base:', BASE_MODEL)

## 7. Merge → GGUF → quantize → push to your HF repo (one call)

In [ ]:
# Unsloth handles: LoRA merge, llama.cpp clone+build, convert to f16 GGUF,
# llama-quantize for each requested method, and upload to the HF repo.
model.push_to_hub_gguf(
    OUTPUT_REPO,
    tokenizer,
    quantization_method = QUANTS,
    token               = HF_TOKEN,
    private             = PRIVATE_REPO,
)
print(f"\nDone → https://huggingface.co/{OUTPUT_REPO}")

## 8. Add a README to the uploaded repo

In [ ]:
from huggingface_hub import HfApi
hack_note = f"hacked_harness rate: **{hr:.3f}**" if hr is not None else "(manually selected step)"
readme = f"""---
base_model: {BASE_MODEL}
tags: [gguf, qwen3, reward-hacking, lora-merged, unsloth]
---

# {TAG} — GGUF

GGUF quantizations of `{BASE_MODEL}` merged with LoRA adapter `{ADAPTER_REPO}` at `{STEP_DIR}`.

Auto-selected as the peak-reward-hacking checkpoint — {hack_note}.

**Deliberately reward-hacked research checkpoint.** Not for production use.

Quants: {', '.join(QUANTS)}

Produced via Unsloth's `push_to_hub_gguf`.
"""
with open("/content/README.md", "w") as f: f.write(readme)
HfApi().upload_file(path_or_fileobj="/content/README.md", path_in_repo="README.md", repo_id=OUTPUT_REPO, token=HF_TOKEN)
print(f"README → https://huggingface.co/{OUTPUT_REPO}")

## 9. (Optional) Run it locally

```bash
# llama.cpp
./llama-cli -m <file>.Q4_K_M.gguf -p 'Write a python function that...'

# Ollama
echo 'FROM ./<file>.Q4_K_M.gguf' > Modelfile
ollama create qwen3-rh -f Modelfile && ollama run qwen3-rh
```

Q4 smooths out subtle hacking behavior — use **Q6_K or F16** if you're studying the hack dynamics.